# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1,2,3] [ 0 1 1 ] // - деление с округлением
2. [4, 4, 4, 4] matrix with 1, sum of the raws  
3. [0 1 2 ] +0
   [1 2 3 ] +1 
   [2 3 4 ] +2
4. 1+2+3 = 6 
5. [1, 2, 3, 4] = 1 4 9 16 = 30/4 = 7.5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "a  stana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))  


amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [13]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid      
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [22]:
# 1  в чистых данных: 4739
clean_count = clean.sum()

september = amount[clean & (month == "2026-09")].sum()
october = amount[clean & (month == "2026-10")].sum()

change = (october - september) / september * 100

print("Чистых заказов:", clean_count)
print(f"Сентябрь: {september:.0f} ₸")
print(f"Октябрь: {october:.0f} ₸")
print(f"Изменение: {change:.1f}%")

Чистых заказов: 4739
Сентябрь: 35243410 ₸
Октябрь: 34419520 ₸
Изменение: -2.3%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [30]:
# 2
categories = np.unique(cat)
months = np.unique(month)

revenue_table = np.zeros((len(categories), len(months)))

for i in range(len(categories)):
    for j in range(len(months)):
        mask = clean & (cat == categories[i]) & (month == months[j])
        revenue_table[i, j] = amount[mask].sum()

print(f"{'Категория':<15} | {'Сентябрь':>12} | {'Октябрь':>12}")

for i in range(len(categories)):
    print(f"{categories[i]:<15} | {revenue_table[i, 0]:>12.0f} | {revenue_table[i, 1]:>12.0f}")

total_revenue = revenue_table.sum(axis=1)
max_index = total_revenue.argmax()

percent_change = (revenue_table[:, 1] - revenue_table[:, 0]) / revenue_table[:, 0] * 100
min_index = percent_change.argmin()

print()
print("Самая прибыльная категория:", categories[max_index])
print("Общая выручка:", total_revenue[max_index], "₸")
print("Самое сильное падение:", categories[min_index])
print(f"Падение: {percent_change[min_index]:.1f}%")

Категория       |     Сентябрь |      Октябрь
Дом             |      4997730 |      5125790
Книги           |      1065300 |      1140910
Красота         |      2610950 |      2678990
Одежда          |      9167100 |      8505730
Электроника     |     17402330 |     16968100

Самая прибыльная категория: Электроника
Общая выручка: 34370430.0 ₸
Самое сильное падение: Одежда
Падение: -7.2%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [37]:
# 3
count_table = np.zeros((len(categories), len(months)))

for i in range(len(categories)):
    for j in range(len(months)):
        mask = clean & (cat == categories[i]) & (month == months[j])
        count_table[i, j] = mask.sum()

average_table = revenue_table / count_table

print(f"{'Категория':<15} | {'Заказов сент.':>14} | {'Заказов окт.':>14} | {'Средний чек окт.':>18}")

for i in range(len(categories)):
    print(f"{categories[i]:<15} | {count_table[i, 0]:>14.0f} | {count_table[i, 1]:>14.0f} | {average_table[i, 1]:>18.0f}")

max_index = average_table[:, 1].argmax()

print()
print("Самый высокий средний чек:", categories[max_index])
print(f"Средний чек: {average_table[max_index, 1]:.0f} ₸")

Категория       |  Заказов сент. |   Заказов окт. |   Средний чек окт.
Дом             |            476 |            476 |              10768
Книги           |            265 |            299 |               3816
Красота         |            422 |            435 |               6159
Одежда          |            678 |            652 |              13046
Электроника     |            531 |            505 |              33600

Самый высокий средний чек: Электроника
Средний чек: 33600 ₸


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [43]:
# 4
cities = np.unique(city)
months = np.unique(month)

table = np.zeros((len(cities), len(months)))

for i in range(len(cities)):
    for j in range(len(months)):
        mask = clean & (city == cities[i]) & (month == months[j])
        table[i, j] = amount[mask].sum()

share = table / table.sum(axis=0)

print(f"{'Город':<12} | {'Сентябрь':>12} | {'Октябрь':>12}")

for i in range(len(cities)):
    print(f"{cities[i]:<12} | {share[i, 0]:>11.1%} | {share[i, 1]:>11.1%}")

print("\nСуммы столбцов:", share.sum(axis=0))

shymkent_index = np.where(cities == "Шымкент")[0][0]

sep_share = share[shymkent_index, 0]
oct_share = share[shymkent_index, 1]

change = (oct_share - sep_share) * 100

print(f"\nШымкент в сентябре: {sep_share:.1%}")
print(f"Шымкент в октябре: {oct_share:.1%}")
print(f"Изменение: {change:.2f} п.п.")

Город        |     Сентябрь |      Октябрь
Astana       |        1.5% |        1.7%
Актобе       |        7.3% |        7.4%
Алматы       |       41.1% |       42.5%
Астана       |       24.2% |       24.0%
Атырау       |        2.6% |        2.8%
Караганда    |       10.4% |        9.8%
Шымкент      |       12.9% |       11.7%

Суммы столбцов: [1. 1.]

Шымкент в сентябре: 12.9%
Шымкент в октябре: 11.7%
Изменение: -1.19 п.п.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [44]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [45]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [46]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [47]:
# 8